# 07 — Integrated Disease Dynamics Analysis

## Objective

Integrate the cleaned **COVID-19**, **Testing**, and **Vaccination** datasets to study how disease burden, testing activity, and vaccination changed together across time and states.

### Main questions

1. How did testing volume relate to detected COVID-19 cases?
2. How did positivity change during the first and second waves?
3. How did vaccination progress relative to COVID-19 cases and deaths?
4. Do states with higher vaccination levels show different subsequent disease outcomes?
5. How are testing, vaccination and COVID-19 burden related at the state level?

> This notebook focuses on association and disease dynamics, not causal inference. Population size, reporting practices, interventions, variants and other confounders are not fully controlled.


In [ ]:
# 1. Imports and load cleaned datasets

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (13, 6)

COVID_PATH = Path("covid_clean.csv")
TESTING_PATH = Path("testing_clean.csv")
VACCINATION_PATH = Path("vaccination_clean.csv")

covid = pd.read_csv(COVID_PATH)
testing = pd.read_csv(TESTING_PATH)
vaccination = pd.read_csv(VACCINATION_PATH)

covid["Date"] = pd.to_datetime(covid["Date"], errors="coerce")
testing["Date"] = pd.to_datetime(testing["Date"], errors="coerce")
vaccination["Updated On"] = pd.to_datetime(
    vaccination["Updated On"], errors="coerce"
)

print("COVID:", covid.shape)
print("Testing:", testing.shape)
print("Vaccination:", vaccination.shape)


## 2. Standardize state names

In [ ]:
# These mappings address known spelling/name inconsistencies found during exploration.
state_map = {
    "Telengana": "Telangana",
    "Karanataka": "Karnataka",
    "Himanchal Pradesh": "Himachal Pradesh",
    "Dadra and Nagar Haveli": "Dadra and Nagar Haveli and Daman and Diu",
    "Daman & Diu": "Dadra and Nagar Haveli and Daman and Diu",
    "Daman and Diu": "Dadra and Nagar Haveli and Daman and Diu",
    "Bihar****": "Bihar",
    "Madhya Pradesh***": "Madhya Pradesh",
    "Maharashtra***": "Maharashtra",
}

covid["State"] = covid["State/UnionTerritory"].replace(state_map)
testing["State"] = testing["State"].replace(state_map)
vaccination["State"] = vaccination["State"].replace(state_map)

# Exclude non-geographic national/reassignment rows where appropriate.
covid = covid[
    ~covid["State"].isin(["India", "Unassigned", "Cases being reassigned to states"])
].copy()

testing = testing[testing["State"] != "India"].copy()

print("COVID states:", covid["State"].nunique())
print("Testing states:", testing["State"].nunique())
print("Vaccination states:", vaccination["State"].nunique())


## 3. Create national daily COVID, testing and vaccination series

In [ ]:
# ---------- COVID ----------
if "NewCases" not in covid.columns:
    covid["NewCases"] = covid.groupby("State")["Confirmed"].diff()

if "NewDeaths" not in covid.columns:
    covid["NewDeaths"] = covid.groupby("State")["Deaths"].diff()

covid_daily = (
    covid.groupby("Date", as_index=False)
    .agg(
        NewCases=("NewCases", "sum"),
        NewDeaths=("NewDeaths", "sum"),
        Confirmed=("Confirmed", "sum"),
        Deaths=("Deaths", "sum"),
        Cured=("Cured", "sum")
    )
    .sort_values("Date")
)

covid_daily["NewCases"] = covid_daily["NewCases"].clip(lower=0)
covid_daily["NewDeaths"] = covid_daily["NewDeaths"].clip(lower=0)
covid_daily["Cases_7D"] = covid_daily["NewCases"].rolling(7, min_periods=1).mean()
covid_daily["Deaths_7D"] = covid_daily["NewDeaths"].rolling(7, min_periods=1).mean()


# ---------- TESTING ----------
testing["NewTests"] = testing.groupby("State")["TotalSamples"].diff()
testing["NewTests"] = testing["NewTests"].where(testing["NewTests"] >= 0)

testing_daily = (
    testing.groupby("Date", as_index=False)
    .agg(
        TotalSamples=("TotalSamples", "sum"),
        NewTests=("NewTests", "sum"),
        Positive=("Positive", "sum")
    )
    .sort_values("Date")
)

testing_daily["NewTests_7D"] = testing_daily["NewTests"].rolling(7, min_periods=1).mean()
testing_daily["DailyPositivity_%"] = (
    testing_daily["Positive"] / testing_daily["TotalSamples"] * 100
)


# ---------- VACCINATION ----------
vaccination["NewDoses"] = (
    vaccination.groupby("State")["Total Doses Administered"].diff()
)

vaccination["NewDoses"] = vaccination["NewDoses"].where(
    vaccination["NewDoses"] >= 0
)

vaccination_daily = (
    vaccination.groupby("Updated On", as_index=False)
    .agg(
        TotalDoses=("Total Doses Administered", "sum"),
        NewDoses=("NewDoses", "sum"),
        FirstDose=("First Dose Administered", "sum"),
        SecondDose=("Second Dose Administered", "sum")
    )
    .rename(columns={"Updated On": "Date"})
    .sort_values("Date")
)

vaccination_daily["NewDoses_7D"] = (
    vaccination_daily["NewDoses"].rolling(7, min_periods=1).mean()
)

display(covid_daily.head())
display(testing_daily.head())
display(vaccination_daily.head())


## 4. Merge the three national datasets

In [ ]:
integrated = covid_daily.merge(
    testing_daily,
    on="Date",
    how="left"
).merge(
    vaccination_daily,
    on="Date",
    how="left"
)

integrated = integrated.sort_values("Date").reset_index(drop=True)

print("Integrated shape:", integrated.shape)
print("Date range:", integrated["Date"].min(), "to", integrated["Date"].max())

display(integrated.head())


## 5. Overall temporal comparison

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(
    integrated["Date"],
    integrated["Cases_7D"],
    label="COVID cases — 7-day average"
)
ax1.set_xlabel("Date")
ax1.set_ylabel("New COVID cases")

ax2 = ax1.twinx()
ax2.plot(
    integrated["Date"],
    integrated["NewDoses_7D"],
    label="New vaccine doses — 7-day average",
    linestyle="--"
)
ax2.set_ylabel("New vaccine doses")

plt.title("COVID-19 Cases and Vaccination Over Time")
fig.tight_layout()
plt.show()


## 6. Testing versus COVID cases

In [ ]:
# Align testing and cases using the same dates.
testing_cases = integrated[
    ["Date", "Cases_7D", "NewTests_7D", "DailyPositivity_%"]
].dropna()

print("Correlation: testing volume vs cases")
print(
    testing_cases["NewTests_7D"].corr(
        testing_cases["Cases_7D"]
    )
)

plt.scatter(
    testing_cases["NewTests_7D"],
    testing_cases["Cases_7D"],
    alpha=0.5
)
plt.xlabel("7-day average new tests")
plt.ylabel("7-day average new COVID cases")
plt.title("Testing Intensity vs Detected COVID-19 Cases")
plt.tight_layout()
plt.show()


## 7. Positivity rate across the waves

In [ ]:
plt.plot(
    integrated["Date"],
    integrated["DailyPositivity_%"],
    label="Cumulative/daily aligned positivity"
)

plt.axvline(pd.Timestamp("2020-09-17"), linestyle="--", label="First-wave peak")
plt.axvline(pd.Timestamp("2021-05-09"), linestyle="--", label="Second-wave peak")

plt.title("Testing Positivity Over the COVID-19 Period")
plt.xlabel("Date")
plt.ylabel("Positivity (%)")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Vaccination relative to the second wave

In [ ]:
second_wave_window = integrated[
    (integrated["Date"] >= "2021-01-01") &
    (integrated["Date"] <= "2021-08-09")
].copy()

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(
    second_wave_window["Date"],
    second_wave_window["Cases_7D"],
    label="COVID cases — 7-day average"
)
ax1.set_ylabel("COVID cases")

ax2 = ax1.twinx()
ax2.plot(
    second_wave_window["Date"],
    second_wave_window["NewDoses_7D"],
    linestyle="--",
    label="New vaccine doses — 7-day average"
)
ax2.set_ylabel("New vaccine doses")

plt.axvline(pd.Timestamp("2021-05-09"), linestyle=":")
plt.title("Vaccination Progress and Second COVID-19 Wave")
fig.tight_layout()
plt.show()


## 9. Lagged relationship: vaccination and subsequent cases

In [ ]:
# Explore whether vaccination activity is associated with cases
# at later dates. This is descriptive and is NOT a causal test.

lag_results = []

for lag in [7, 14, 21, 28, 42, 56]:
    temp = integrated[["Date", "NewDoses_7D", "Cases_7D"]].copy()
    temp["Future_Cases"] = temp["Cases_7D"].shift(-lag)

    corr = temp["NewDoses_7D"].corr(temp["Future_Cases"])

    lag_results.append({
        "Lag_days": lag,
        "Correlation_vaccination_future_cases": corr
    })

lag_df = pd.DataFrame(lag_results)

display(lag_df)

plt.plot(
    lag_df["Lag_days"],
    lag_df["Correlation_vaccination_future_cases"],
    marker="o"
)
plt.axhline(0, linestyle="--")
plt.xlabel("Lag from vaccination activity (days)")
plt.ylabel("Correlation")
plt.title("Vaccination Activity vs Future COVID Cases")
plt.tight_layout()
plt.show()


## 10. State-level integrated dataset

In [ ]:
# COVID state summary
covid_state = (
    covid.groupby("State", as_index=False)
    .agg(
        Confirmed=("Confirmed", "max"),
        Deaths=("Deaths", "max"),
        Cured=("Cured", "max")
    )
)

covid_state["CFR_%"] = (
    covid_state["Deaths"] /
    covid_state["Confirmed"].replace(0, np.nan) * 100
)


# Testing state summary
testing_state = (
    testing.groupby("State", as_index=False)
    .agg(
        TotalSamples=("TotalSamples", "max"),
        Positive=("Positive", "max")
    )
)

testing_state["Positivity_%"] = (
    testing_state["Positive"] /
    testing_state["TotalSamples"].replace(0, np.nan) * 100
)


# Vaccination state summary
vaccination_state = (
    vaccination.groupby("State", as_index=False)
    .agg(
        TotalDoses=("Total Doses Administered", "max"),
        FirstDose=("First Dose Administered", "max"),
        SecondDose=("Second Dose Administered", "max")
    )
)

state_integrated = (
    covid_state
    .merge(testing_state, on="State", how="inner")
    .merge(vaccination_state, on="State", how="inner")
)

state_integrated["SecondDoseShare_%"] = (
    state_integrated["SecondDose"] /
    state_integrated["FirstDose"].replace(0, np.nan) * 100
)

display(state_integrated.sort_values("Confirmed", ascending=False).head(15))
print("Integrated states:", state_integrated["State"].nunique())


## 11. Testing volume versus COVID burden by state

In [ ]:
plt.scatter(
    state_integrated["TotalSamples"],
    state_integrated["Confirmed"]
)

plt.xlabel("Total samples")
plt.ylabel("Cumulative confirmed cases")
plt.title("State-Level Testing Volume vs COVID-19 Cases")
plt.tight_layout()
plt.show()

print(
    "Correlation:",
    state_integrated["TotalSamples"].corr(
        state_integrated["Confirmed"]
    )
)


## 12. Vaccination versus cumulative COVID burden

In [ ]:
plt.scatter(
    state_integrated["TotalDoses"],
    state_integrated["Confirmed"]
)

plt.xlabel("Total vaccine doses")
plt.ylabel("Cumulative confirmed cases")
plt.title("State-Level Vaccination vs Cumulative COVID-19 Cases")
plt.tight_layout()
plt.show()

print(
    "Correlation:",
    state_integrated["TotalDoses"].corr(
        state_integrated["Confirmed"]
    )
)


## 13. Vaccination versus COVID deaths

In [ ]:
plt.scatter(
    state_integrated["TotalDoses"],
    state_integrated["Deaths"]
)

plt.xlabel("Total vaccine doses")
plt.ylabel("Cumulative deaths")
plt.title("State-Level Vaccination vs COVID-19 Deaths")
plt.tight_layout()
plt.show()

print(
    "Correlation:",
    state_integrated["TotalDoses"].corr(
        state_integrated["Deaths"]
    )
)


## 14. Vaccination and testing together

In [ ]:
# A simple three-variable correlation matrix.
corr_cols = [
    "Confirmed",
    "Deaths",
    "TotalSamples",
    "Positivity_%",
    "TotalDoses",
    "FirstDose",
    "SecondDose"
]

corr_matrix = state_integrated[corr_cols].corr()

display(corr_matrix)

plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, aspect="auto")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar(label="Correlation")
plt.title("State-Level Integrated Correlation Matrix")
plt.tight_layout()
plt.show()


## 15. Identify high-burden states

In [ ]:
top_cases = state_integrated.sort_values(
    "Confirmed", ascending=False
).head(10)

top_deaths = state_integrated.sort_values(
    "Deaths", ascending=False
).head(10)

top_positivity = state_integrated.sort_values(
    "Positivity_%", ascending=False
).head(10)

top_vaccination = state_integrated.sort_values(
    "TotalDoses", ascending=False
).head(10)

print("Top 10 by confirmed cases")
display(top_cases[[
    "State", "Confirmed", "Deaths",
    "TotalSamples", "Positivity_%", "TotalDoses"
]])

print("Top 10 by deaths")
display(top_deaths[[
    "State", "Confirmed", "Deaths",
    "TotalDoses"
]])

print("Top 10 by testing positivity")
display(top_positivity[[
    "State", "TotalSamples", "Positive", "Positivity_%"
]])

print("Top 10 by vaccination")
display(top_vaccination[[
    "State", "TotalDoses", "FirstDose", "SecondDose"
]])


## 16. Optional population-normalized analysis

In [ ]:
# If you have a state population CSV, merge it here.
#
# Expected columns:
#   State
#   Population
#
# Example:
# population = pd.read_csv("state_population.csv")
# state_integrated = state_integrated.merge(population, on="State", how="left")
#
# state_integrated["Cases_per_100k"] = (
#     state_integrated["Confirmed"] / state_integrated["Population"] * 100000
# )
#
# state_integrated["Deaths_per_100k"] = (
#     state_integrated["Deaths"] / state_integrated["Population"] * 100000
# )
#
# state_integrated["Doses_per_100k"] = (
#     state_integrated["TotalDoses"] / state_integrated["Population"] * 100000
# )
#
# display(state_integrated.sort_values("Cases_per_100k", ascending=False).head(10))

print("Population normalization is recommended before making claims that one state performed better than another.")


## 17. Main integrated findings to report

In [ ]:
print("""When writing the final report, focus on these findings:

1. TEMPORAL DYNAMICS
   - Compare the first and second COVID waves.
   - Describe how testing and positivity changed during each wave.
   - Describe the timing and acceleration of vaccination.

2. TESTING → DETECTION
   - Use the testing/case correlation carefully.
   - Higher testing can increase detected cases, so correlation is not causation.

3. VACCINATION → DISEASE BURDEN
   - Compare vaccination progress with subsequent cases/deaths.
   - Use lagged relationships rather than only same-day comparisons.

4. SPATIAL HETEROGENEITY
   - Identify states with high cases, deaths, testing, positivity and vaccination.
   - Avoid interpreting raw totals without population adjustment.

5. LIMITATIONS
   - Reporting revisions
   - Missing optional vaccination variables
   - Unequal testing capacity
   - Population differences
   - Variant/intervention effects
   - Observational data cannot establish causality
""")


## 18. Save integrated outputs

The following files can be used later for the final report/dashboard.


In [ ]:
integrated.to_csv("integrated_national_daily.csv", index=False)
state_integrated.to_csv("integrated_state_summary.csv", index=False)
lag_df.to_csv("vaccination_case_lag_correlations.csv", index=False)

print("Saved:")
print("- integrated_national_daily.csv")
print("- integrated_state_summary.csv")
print("- vaccination_case_lag_correlations.csv")
